# 01 — Exploratory Data Analysis (EuroSAT RGB)
**Owner: Member 1 — Data Engineer** · Milestone 1+2 deliverable.

Audit the dataset, measure imbalance, check duplicates, and end with the findings → modelling decisions list required by AIE231 spec 3.1. This notebook only *presents* results; all logic imports from `src/`.

In [ ]:
import sys
from pathlib import Path
if not Path("configs").exists(): sys.path.insert(0, str(Path.cwd().parent))
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

from configs import config as cfg
from src import data as data_mod

DATA_DIR = cfg.DATA_DIR   # e.g. 'data/EuroSAT' after data/download_data.py
print(cfg.EUROSAT_CITATION)

## 1. Class distribution — who is the majority?

In [ ]:
paths, labels, names = data_mod.scan_dataset(DATA_DIR)
counts = Counter(labels.tolist())
dist = {names[i]: counts.get(i, 0) for i in range(len(names))}

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(dist.keys(), dist.values(), color='#4A7A8C')
ax.set_title('EuroSAT RGB — images per class (full dataset: 27,000)')
ax.set_ylabel('images')
plt.xticks(rotation=45, ha='right')
ax.grid(alpha=0.3, axis='y')
plt.show()

ratio = max(dist.values()) / min(dist.values())
print(f'moderate imbalance ratio = {ratio:.2f} (max {max(dist, key=dist.get)} / min {min(dist, key=dist.get)})')

## 2. Sample tiles per class — what do the classes look like?

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, cname in zip(axes.flat, names):
    folder = Path(DATA_DIR) / cname
    img = plt.imread(sorted(folder.glob('*.jpg'))[0])
    ax.imshow(img); ax.set_title(cname, fontsize=9); ax.axis('off')
fig.suptitle('One tile per class (64x64 Sentinel-2 RGB)')
plt.show()

## 3. Audit: image sizes, duplicates, split leakage

In [ ]:
import json
audit = data_mod.audit_dataset(DATA_DIR)
print(json.dumps(audit, indent=2))

In [ ]:
splits = data_mod.stratified_split(paths, labels)
data_mod.leakage_check(splits)          # raises if any file appears twice
print('leakage check passed ✔')
for k, (p, y) in splits.items():
    print(f'{k:>6}: {len(p):>6} images  {data_mod.class_distribution(y)}')

## 4. Class weights for the imbalance (spec 3.2)

In [ ]:
weights = data_mod.compute_class_weights(splits['train'][1])
for i, (n, w) in enumerate(zip(names, weights.values())):
    print(f'{n:<22} weight={w:.2f}')

In [ ]:
print(f"Class imbalance max/min: {ratio:.2f}; use class weights, compare per-class recall.")
print(f"Duplicate groups: {audit['duplicate_groups_full_scan']}; deduplicate before splitting.")
print(f"Corrupt files: {audit['corrupt_files']}; exclude before splitting.")
print(f"Image shapes: {audit['sampled_image_shapes']}; resize to 64x64 RGB.")
print("Orientation may vary; compare training-only flips/rotations/zoom/contrast with no augmentation.")
print("Scene metadata not provided; near-duplicate/geographic leakage remains a limitation.")